# Price Determining Function

Creating a simple function that determines price according to merit order data.

In [7]:
# Let's load in the necessary packages
import pandas as pd
import datetime
import os
import timeit
import matplotlib.pyplot as plt
from ortools.linear_solver import pywraplp
# from ortools.sat.python import cp_model
import numpy as np

#### Function from Prof. Leach
Will use here the code that Professor Leach gave.
It's a function that gets the new price for a certain hour.

In [10]:
# Loading in the data first.
# Let's load in the data here, and for now, let's make it just for four hours.

file_name = os.fsdecode("marco_data.csv")
workbook = pd.read_csv(file_name)
workbook = workbook.filter(items=["date", "he", "size","flexible","price"]) # Gets the columns that we need

workbook = workbook.query('date=="2024-04-10"','hour<=18', 'hour>=15') # Remove this to take out the filter for the day.

# Getting the columns we need.
merit_data = workbook.filter(items=["date", "he", "size","flexible","price","ail"]) # Gets the columns that we need

pd.set_option('display.max_rows',50)

# Temporary peek:
temp = merit_data.copy()
temp = temp.query('he == 15')
temp


FileNotFoundError: [Errno 2] No such file or directory: 'marco_data.csv'

In [ ]:
# Making everything we have above into a function:

def pricing(bids, ail): # Bids is self-explanatory, so set of bids needed.
    bids=bids.sort_values('price')
    # bids=bids.assign(merit=bids['size'].cumsum())
    bids['merit'] = bids['size'].cumsum() # Old code is above
    # bids = bids.assign(surplus=bids['merit']-bids['ail'].max())
    bids['surplus'] = bids['merit'] - bids['ail']
    last_full=bids.query('surplus>0')['surplus'].idxmin()
    # Made this change to get the actual energy needed left
    e_needed=-bids['surplus'][last_full-1] # Getting the last negative surplus value since that's the amound lacking
    while(e_needed>0):
        if(bids['size'][last_full]<e_needed): #dispatch it
            e_needed-=size
            last_full+=1
        elif (bids['size'][last_full]>e_needed) and (bids['flexible'][last_full]=="Y"): #dispatch part of it 
            e_needed-=e_needed
            last_full+=1
        # Made changes in the bids below to get the price accurately
        price=bids['price'][last_full-1]
        print(e_needed,last_full-1,price)


In [4]:
# Trying out our function

pricing(merit_data.query('he == 15'),merit_data['ail'].max())

NameError: name 'pricing' is not defined